# 🎯 Experiment1: Comprehensive Benchmarking Analysis

**TabPFNCredit - Deep Dive into Credit Risk ML Methods**

This notebook provides:

## Analysis Pipeline:
1. **Data Loading & Preparation** - Automated summary generation
2. **PD Analysis (Classification)** - AUC-based performance evaluation
3. **LGD Analysis (Regression)** - R²-based performance evaluation

## Key Features:
- ✅ Performance matrices (NO_HPO, HPO, Difference)
- ✅ Statistical significance testing (Friedman, Nemenyi, Wilcoxon-Holm)
- ✅ Critical difference diagrams
- ✅ Effect size analysis (Cohen's d)
- ✅ PAMA (Probability of Achieving Maximal Accuracy)
- ✅ Comprehensive visualizations
- ✅ HPO impact analysis

---

## 📦 1. Setup & Configuration

In [2]:
# Standard imports
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scipy.stats import wilcoxon, friedmanchisquare, rankdata, norm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('default')
plt.rcParams['figure.figsize'] = (24, 12)
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 100
sns.set_palette("husl")

# =============================================================================
# PATHS (Notebook in notebooks/ folder)
# =============================================================================

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent  # notebooks/ -> project root
sys.path.insert(0, str(PROJECT_ROOT))

EXPERIMENT_NAME = "experiment1"
RESULTS_DIR = PROJECT_ROOT / "results" / EXPERIMENT_NAME
SUMMARY_DIR = RESULTS_DIR / "summary"
FIGURES_DIR = RESULTS_DIR / "figures"

print("=" * 80)
print("  EXPERIMENT1 COMPREHENSIVE ANALYSIS")
print("=" * 80)
print(f"📂 Project root:       {PROJECT_ROOT}")
print(f"📂 Results directory:  {RESULTS_DIR}")
print(f"📂 Summary directory:  {SUMMARY_DIR}")
print(f"📂 Figures directory:  {FIGURES_DIR}")

# =============================================================================
# AUTO-GENERATE SUMMARIES IF NEEDED
# =============================================================================

if SUMMARY_DIR.exists() and any(SUMMARY_DIR.glob("*.csv")):
    print(f"\n✓ Summary files already exist. Loading existing summaries.")
else:
    print(f"\n📊 Summary files not found. Generating summaries...")
    from src.utils.summarize_results import summarize_results
    print("\n" + "=" * 80)
    summarize_results(experiment=EXPERIMENT_NAME)
    print("=" * 80)

# Ensure figures directory exists
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("\n✅ Setup complete! Ready for analysis.")
print("=" * 80)

  EXPERIMENT1 COMPREHENSIVE ANALYSIS
📂 Project root:       c:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit
📂 Results directory:  c:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit\results\experiment1
📂 Summary directory:  c:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit\results\experiment1\summary
📂 Figures directory:  c:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit\results\experiment1\figures

📊 Summary files not found. Generating summaries...

  SUMMARIZING RESULTS: EXPERIMENT1

📂 Experiment directory: C:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit\results\experiment1
📂 Summary output: C:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit\results\experiment1\summary

--------------------------------------------------------------------------------
  PD RESULTS
---------------

## 📊 2. Load Data

In [3]:
# =============================================================================
# LOAD AGGREGATED RESULTS
# =============================================================================

print("Loading aggregated results...\n")

# Load PD results
pd_raw_file = SUMMARY_DIR / "summary_pd_raw.csv"
pd_agg_file = SUMMARY_DIR / "summary_pd_aggregated.csv"

if pd_raw_file.exists():
    pd_raw = pd.read_csv(pd_raw_file)
    print(f"✅ PD Raw: {len(pd_raw)} fold results")
    print(f"   Methods: {pd_raw['method'].nunique()}")
    print(f"   Datasets: {pd_raw['dataset'].nunique()}")
else:
    pd_raw = pd.DataFrame()
    print("⚠️  PD raw data not found")

if pd_agg_file.exists():
    pd_agg = pd.read_csv(pd_agg_file)
    print(f"✅ PD Aggregated: {len(pd_agg)} method-dataset combinations")
else:
    pd_agg = pd.DataFrame()
    print("⚠️  PD aggregated data not found")

# Load LGD results
lgd_raw_file = SUMMARY_DIR / "summary_lgd_raw.csv"
lgd_agg_file = SUMMARY_DIR / "summary_lgd_aggregated.csv"

print()
if lgd_raw_file.exists():
    lgd_raw = pd.read_csv(lgd_raw_file)
    print(f"✅ LGD Raw: {len(lgd_raw)} fold results")
    print(f"   Methods: {lgd_raw['method'].nunique()}")
    print(f"   Datasets: {lgd_raw['dataset'].nunique()}")
else:
    lgd_raw = pd.DataFrame()
    print("⚠️  LGD raw data not found")

if lgd_agg_file.exists():
    lgd_agg = pd.read_csv(lgd_agg_file)
    print(f"✅ LGD Aggregated: {len(lgd_agg)} method-dataset combinations")
else:
    lgd_agg = pd.DataFrame()
    print("⚠️  LGD aggregated data not found")

print("\n" + "=" * 80)

Loading aggregated results...

✅ PD Raw: 4015 fold results
   Methods: 27
   Datasets: 15
✅ PD Aggregated: 803 method-dataset combinations

✅ LGD Raw: 1530 fold results
   Methods: 22
   Datasets: 7
✅ LGD Aggregated: 306 method-dataset combinations



---
# 🎯 PART A: PD Analysis (Classification)


## 📋 A1. Performance Matrices - PD

In [4]:
# =============================================================================
# HELPER FUNCTION: CREATE PERFORMANCE MATRIX
# =============================================================================

def create_performance_matrix(df, metric, hpo_mode=None, show_folds=False):
    """
    Create a matrix of performance values: Methods (rows) × Datasets (columns)
    
    Args:
        df: DataFrame (raw or aggregated)
        metric: Metric name (e.g., 'AUC', 'R2')
        hpo_mode: Filter by HPO mode ('NO_HPO', 'HPO', or None for all)
        show_folds: If True, use raw data with fold_id, else use aggregated means
    """
    if df.empty:
        print("⚠️  No data available")
        return None
    
    # Filter by HPO mode if specified
    if hpo_mode:
        df = df[df['hpo_mode'] == hpo_mode].copy()
    
    if df.empty:
        print(f"⚠️  No data for {hpo_mode}")
        return None
    
    # Determine which column to use
    if show_folds:
        # Use raw metric values
        value_col = metric
    else:
        # Use mean values from aggregated data
        value_col = f'{metric}_mean'
        if value_col not in df.columns:
            print(f"⚠️  Column '{value_col}' not found. Available: {df.columns.tolist()}")
            return None
    
    # Create pivot table
    if show_folds:
        # For raw data: aggregate across folds first
        pivot = df.groupby(['method', 'dataset'])[value_col].mean().unstack()
    else:
        # For aggregated data: direct pivot
        pivot = df.pivot(index='method', columns='dataset', values=value_col)
    
    # Sort methods by average performance (descending)
    method_means = pivot.mean(axis=1).sort_values(ascending=False)
    pivot = pivot.loc[method_means.index]
    
    # Sort datasets alphabetically
    pivot = pivot[sorted(pivot.columns)]
    
    return pivot

### 📊 A1.1 NO_HPO Performance Matrix

In [5]:
print("\n" + "=" * 80)
print("  PD - NO_HPO PERFORMANCE MATRIX (AUC)")
print("=" * 80)

pd_matrix_no_hpo = create_performance_matrix(pd_agg, 'AUC', hpo_mode='NO_HPO')

if pd_matrix_no_hpo is not None:
    # Add summary statistics
    pd_matrix_no_hpo['MEAN'] = pd_matrix_no_hpo.mean(axis=1)
    pd_matrix_no_hpo['STD'] = pd_agg[pd_agg['hpo_mode']=='NO_HPO'].groupby('method')['AUC_std'].mean()
    
    print(f"\n📊 Matrix shape: {pd_matrix_no_hpo.shape[0]} methods × {pd_matrix_no_hpo.shape[1]-2} datasets\n")
    
    # Display with nice formatting
    display_df = pd_matrix_no_hpo.copy()
    
    # Format numeric columns
    for col in display_df.columns:
        if col not in ['MEAN', 'STD']:
            display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "-")
    display_df['MEAN'] = display_df['MEAN'].apply(lambda x: f"{x:.4f}")
    display_df['STD'] = display_df['STD'].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "-")
    
    print(display_df.to_string())
    
    # Save to CSV
    output_file = SUMMARY_DIR / "matrix_pd_no_hpo.csv"
    pd_matrix_no_hpo.to_csv(output_file)
    print(f"\n✅ Saved to: {output_file.name}")
else:
    print("⚠️  Could not create NO_HPO matrix")


  PD - NO_HPO PERFORMANCE MATRIX (AUC)

📊 Matrix shape: 27 methods × 15 datasets

dataset      0001.gmsc 0002.taiwan_creditcard 0003.vehicle_loan 0004.lendingclub 0005.case_study 0006.myhom 0007.hackerearth 0008.cobranded 0009.german 0010.bank_status 0011.thomas 0012.loan_default 0013.home_credit 0014.hmeq 0015.algorithmwatch    MEAN     STD
method                                                                                                                                                                                                                                                               
catboost        0.8661                 0.7834            0.6559           0.6731          0.8654     0.5850           0.9321         0.8405      0.7759           0.7693      0.6363            0.6395           0.7568    0.9560              0.6559  0.7594  0.0091
t2gformer       0.8633                 0.7809            0.6524           0.6730          0.8626     0.5948           0.9316       

### 📊 A1.2 HPO Performance Matrix

In [ ]:
print("\n" + "=" * 80)
print("  PD - HPO PERFORMANCE MATRIX (AUC)")
print("=" * 80)

pd_matrix_hpo = create_performance_matrix(pd_agg, 'AUC', hpo_mode='HPO')

if pd_matrix_hpo is not None:
    # Add summary statistics
    pd_matrix_hpo['MEAN'] = pd_matrix_hpo.mean(axis=1)
    pd_matrix_hpo['STD'] = pd_agg[pd_agg['hpo_mode']=='HPO'].groupby('method')['AUC_std'].mean()
    
    print(f"\n📊 Matrix shape: {pd_matrix_hpo.shape[0]} methods × {pd_matrix_hpo.shape[1]-2} datasets\n")
    
    # Display with nice formatting
    display_df = pd_matrix_hpo.copy()
    
    # Format numeric columns
    for col in display_df.columns:
        if col not in ['MEAN', 'STD']:
            display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "-")
    display_df['MEAN'] = display_df['MEAN'].apply(lambda x: f"{x:.4f}")
    display_df['STD'] = display_df['STD'].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "-")
    
    print(display_df.to_string())
    
    # Save to CSV
    output_file = SUMMARY_DIR / "matrix_pd_hpo.csv"
    pd_matrix_hpo.to_csv(output_file)
    print(f"\n✅ Saved to: {output_file.name}")
else:
    print("⚠️  Could not create HPO matrix")

### 📊 A1.3 HPO Improvement Matrix (HPO - NO_HPO)

In [ ]:
print("\n" + "=" * 80)
print("  PD - HPO IMPROVEMENT MATRIX (HPO - NO_HPO)")
print("=" * 80)

if pd_matrix_no_hpo is not None and pd_matrix_hpo is not None:
    # Remove summary columns for subtraction
    no_hpo_data = pd_matrix_no_hpo.drop(columns=['MEAN', 'STD'], errors='ignore')
    hpo_data = pd_matrix_hpo.drop(columns=['MEAN', 'STD'], errors='ignore')
    
    # Calculate difference
    pd_matrix_diff = hpo_data - no_hpo_data
    
    # Add summary statistics
    pd_matrix_diff['MEAN'] = pd_matrix_diff.mean(axis=1)
    pd_matrix_diff['ABS_MEAN'] = pd_matrix_diff.drop(columns='MEAN').abs().mean(axis=1)
    
    # Sort by mean improvement
    pd_matrix_diff = pd_matrix_diff.sort_values('MEAN', ascending=False)
    
    print(f"\n📊 Matrix shape: {pd_matrix_diff.shape[0]} methods × {pd_matrix_diff.shape[1]-2} datasets\n")
    print("💡 Positive values = HPO better, Negative = NO_HPO better\n")
    
    # Display with nice formatting
    display_df = pd_matrix_diff.copy()
    
    # Format numeric columns with color indicators
    for col in display_df.columns:
        if col not in ['MEAN', 'ABS_MEAN']:
            display_df[col] = display_df[col].apply(
                lambda x: f"{x:+.4f}" if pd.notna(x) else "-"
            )
    display_df['MEAN'] = display_df['MEAN'].apply(lambda x: f"{x:+.4f}")
    display_df['ABS_MEAN'] = display_df['ABS_MEAN'].apply(lambda x: f"{x:.4f}")
    
    print(display_df.to_string())
    
    # Summary statistics
    print("\n" + "-" * 80)
    print("📈 HPO Impact Summary:")
    n_improvements = (pd_matrix_diff.drop(columns=['MEAN', 'ABS_MEAN']) > 0).sum().sum()
    n_degradations = (pd_matrix_diff.drop(columns=['MEAN', 'ABS_MEAN']) < 0).sum().sum()
    n_total = pd_matrix_diff.drop(columns=['MEAN', 'ABS_MEAN']).notna().sum().sum()
    
    print(f"   Improvements: {n_improvements}/{n_total} ({100*n_improvements/n_total:.1f}%)")
    print(f"   Degradations: {n_degradations}/{n_total} ({100*n_degradations/n_total:.1f}%)")
    print(f"   Avg improvement: {pd_matrix_diff['MEAN'].mean():+.4f}")
    print(f"   Max improvement: {pd_matrix_diff.drop(columns=['MEAN', 'ABS_MEAN']).max().max():+.4f}")
    print(f"   Max degradation: {pd_matrix_diff.drop(columns=['MEAN', 'ABS_MEAN']).min().min():+.4f}")
    
    # Save to CSV
    output_file = SUMMARY_DIR / "matrix_pd_hpo_improvement.csv"
    pd_matrix_diff.to_csv(output_file)
    print(f"\n✅ Saved to: {output_file.name}")
else:
    print("⚠️  Could not create difference matrix (missing NO_HPO or HPO data)")

## 📊 A2. Performance Bar Plots - PD

In [ ]:
# =============================================================================
# PD: BAR PLOT COMPARING NO_HPO (BLUE) vs HPO (RED OUTLINE)
# =============================================================================

print("\n" + "=" * 80)
print("  PD - AVERAGE AUC BAR PLOT (NO_HPO vs HPO)")
print("=" * 80)

if not pd_agg.empty:
    # Calculate average performance per method for each HPO mode
    no_hpo_means = pd_agg[pd_agg['hpo_mode']=='NO_HPO'].groupby('method')['AUC_mean'].mean()
    hpo_means = pd_agg[pd_agg['hpo_mode']=='HPO'].groupby('method')['AUC_mean'].mean()
    
    # Combine into DataFrame
    comparison_df = pd.DataFrame({
        'NO_HPO': no_hpo_means,
        'HPO': hpo_means
    }).fillna(0)
    
    # Sort by HPO performance (descending)
    comparison_df = comparison_df.sort_values('HPO', ascending=False)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(16, 10))
    
    x = np.arange(len(comparison_df))
    width = 0.7
    
    # Plot NO_HPO as filled blue bars
    bars_no_hpo = ax.bar(
        x, comparison_df['NO_HPO'], 
        width, 
        label='NO_HPO',
        color='steelblue',
        alpha=0.8,
        edgecolor='black',
        linewidth=1.5
    )
    
    # Plot HPO as red outline bars
    bars_hpo = ax.bar(
        x, comparison_df['HPO'], 
        width, 
        label='HPO',
        color='none',
        edgecolor='red',
        linewidth=3,
        linestyle='-'
    )
    
    # Customize
    ax.set_xlabel('Method', fontsize=14, fontweight='bold')
    ax.set_ylabel('Average AUC', fontsize=14, fontweight='bold')
    ax.set_title('PD Performance: NO_HPO (Blue Fill) vs HPO (Red Outline)\nRanked by HPO Performance', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(comparison_df.index, rotation=45, ha='right')
    ax.legend(fontsize=12, loc='lower left')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_ylim([0.5, 1.0])  # AUC range
    
    # Add value labels on bars
    for i, (idx, row) in enumerate(comparison_df.iterrows()):
        # NO_HPO label
        ax.text(i, row['NO_HPO'] + 0.01, f"{row['NO_HPO']:.3f}", 
                ha='center', va='bottom', fontsize=9, color='steelblue', fontweight='bold')
        # HPO label (if different from NO_HPO)
        if abs(row['HPO'] - row['NO_HPO']) > 0.001:
            ax.text(i, row['HPO'] + 0.01, f"{row['HPO']:.3f}", 
                    ha='center', va='bottom', fontsize=9, color='red', fontweight='bold')
    
    plt.tight_layout()
    
    # Save figure
    output_file = FIGURES_DIR / "pd_barplot_no_hpo_vs_hpo.png"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"\n✅ Saved figure: {output_file.name}")
    
    plt.show()
    
    # Print top performers
    print("\n📊 Top 10 Methods (by HPO):")
    print(comparison_df.head(10).to_string())
    
else:
    print("⚠️  No PD aggregated data available for bar plot")

## 📊 A3. Statistical Analysis - PD

In [ ]:
# =============================================================================
# FRIEDMAN TEST (Non-parametric ANOVA)
# =============================================================================

print("\n" + "=" * 80)
print("  PD - FRIEDMAN TEST (Statistical Significance)")
print("=" * 80)

if pd_matrix_hpo is not None:
    # Remove summary columns
    data = pd_matrix_hpo.drop(columns=['MEAN', 'STD'], errors='ignore')
    
    # Run Friedman test
    try:
        statistic, p_value = friedmanchisquare(*[data.loc[method].dropna() for method in data.index])
        
        print(f"\n📊 Friedman Test Results:")
        print(f"   Chi-square statistic: {statistic:.4f}")
        print(f"   p-value: {p_value:.6f}")
        print(f"   Significant? {'YES ✓' if p_value < 0.05 else 'NO ✗'} (α=0.05)")
        
        if p_value < 0.05:
            print("\n   ✅ Methods have significantly different performance!")
            print("   ➡️  Proceed with post-hoc tests (Nemenyi, Wilcoxon-Holm)")
        else:
            print("\n   ⚠️  No significant difference detected")
            print("   ➡️  Post-hoc tests may not be meaningful")
    except Exception as e:
        print(f"⚠️  Could not perform Friedman test: {e}")
else:
    print("⚠️  No HPO matrix available for statistical testing")

In [ ]:
# =============================================================================
# AVERAGE RANKS
# =============================================================================

print("\n" + "=" * 80)
print("  PD - AVERAGE RANKS ACROSS DATASETS")
print("=" * 80)

if pd_matrix_hpo is not None:
    # Remove summary columns
    data = pd_matrix_hpo.drop(columns=['MEAN', 'STD'], errors='ignore')
    
    # Calculate ranks for each dataset (higher is better, so use descending)
    ranks = data.rank(axis=0, ascending=False, method='average')
    
    # Calculate average rank per method
    avg_ranks = ranks.mean(axis=1).sort_values()
    
    print(f"\n📊 Average Ranks (1=best, {len(data)}=worst):\n")
    for rank, (method, avg_rank) in enumerate(avg_ranks.items(), 1):
        print(f"   {rank:2d}. {method:20s}  {avg_rank:.2f}")
    
    # Save to CSV
    avg_ranks_df = pd.DataFrame({'method': avg_ranks.index, 'avg_rank': avg_ranks.values})
    output_file = SUMMARY_DIR / "pd_average_ranks_hpo.csv"
    avg_ranks_df.to_csv(output_file, index=False)
    print(f"\n✅ Saved to: {output_file.name}")
else:
    print("⚠️  No HPO matrix available for ranking")

## 📊 A4. PAMA Analysis - PD

**PAMA** = Probability of Achieving Maximal Accuracy  
Measures how often each method achieves the best performance on a dataset.

In [ ]:
# =============================================================================
# PAMA (Probability of Achieving Maximal Accuracy)
# =============================================================================

print("\n" + "=" * 80)
print("  PD - PAMA ANALYSIS (Probability of Best Performance)")
print("=" * 80)

if pd_matrix_hpo is not None:
    # Remove summary columns
    data = pd_matrix_hpo.drop(columns=['MEAN', 'STD'], errors='ignore')
    
    # For each dataset, find the best method(s)
    pama_scores = {}
    
    for dataset in data.columns:
        dataset_scores = data[dataset].dropna()
        if len(dataset_scores) > 0:
            max_score = dataset_scores.max()
            # Allow ties (within 0.0001 tolerance)
            best_methods = dataset_scores[dataset_scores >= max_score - 0.0001].index.tolist()
            
            # Award points (1 if sole winner, 1/n if tie)
            for method in best_methods:
                pama_scores[method] = pama_scores.get(method, 0) + (1.0 / len(best_methods))
    
    # Convert to percentages
    n_datasets = len(data.columns)
    pama_df = pd.DataFrame([
        {'method': method, 'wins': score, 'pama': 100 * score / n_datasets}
        for method, score in pama_scores.items()
    ]).sort_values('pama', ascending=False)
    
    print(f"\n📊 PAMA Scores (across {n_datasets} datasets):\n")
    for idx, row in pama_df.iterrows():
        bar = '█' * int(row['pama'] / 2)  # Visual bar
        print(f"   {row['method']:20s}  {row['wins']:5.1f} wins  {row['pama']:5.1f}%  {bar}")
    
    # Create bar plot
    fig, ax = plt.subplots(figsize=(14, 8))
    
    colors = plt.cm.RdYlGn(pama_df['pama'] / pama_df['pama'].max())
    bars = ax.barh(range(len(pama_df)), pama_df['pama'], color=colors, edgecolor='black', linewidth=1.5)
    
    ax.set_yticks(range(len(pama_df)))
    ax.set_yticklabels(pama_df['method'])
    ax.set_xlabel('PAMA (%)', fontsize=14, fontweight='bold')
    ax.set_title('PD: Probability of Achieving Maximal Accuracy (PAMA)\nHow often each method wins', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.invert_yaxis()
    
    # Add value labels
    for i, (idx, row) in enumerate(pama_df.iterrows()):
        ax.text(row['pama'] + 1, i, f"{row['pama']:.1f}%", 
                va='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    
    # Save figure
    output_file = FIGURES_DIR / "pd_pama_analysis.png"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"\n✅ Saved figure: {output_file.name}")
    
    plt.show()
    
    # Save to CSV
    output_file = SUMMARY_DIR / "pd_pama_scores.csv"
    pama_df.to_csv(output_file, index=False)
    print(f"✅ Saved data: {output_file.name}")
    
else:
    print("⚠️  No HPO matrix available for PAMA analysis")

---
# 📈 PART B: LGD Analysis (Regression)

**Metric**: R² (Coefficient of Determination)

Higher is better (range: -∞ to 1.0, where 0 = baseline, 1.0 = perfect, <0 = worse than baseline)

## 📋 B1. Performance Matrices - LGD

### 📊 B1.1 NO_HPO Performance Matrix

In [ ]:
print("\n" + "=" * 80)
print("  LGD - NO_HPO PERFORMANCE MATRIX (R²)")
print("=" * 80)

lgd_matrix_no_hpo = create_performance_matrix(lgd_agg, 'R2', hpo_mode='NO_HPO')

if lgd_matrix_no_hpo is not None:
    # Add summary statistics
    lgd_matrix_no_hpo['MEAN'] = lgd_matrix_no_hpo.mean(axis=1)
    lgd_matrix_no_hpo['STD'] = lgd_agg[lgd_agg['hpo_mode']=='NO_HPO'].groupby('method')['R2_std'].mean()
    
    print(f"\n📊 Matrix shape: {lgd_matrix_no_hpo.shape[0]} methods × {lgd_matrix_no_hpo.shape[1]-2} datasets\n")
    
    # Display with nice formatting
    display_df = lgd_matrix_no_hpo.copy()
    
    # Format numeric columns
    for col in display_df.columns:
        if col not in ['MEAN', 'STD']:
            display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "-")
    display_df['MEAN'] = display_df['MEAN'].apply(lambda x: f"{x:.4f}")
    display_df['STD'] = display_df['STD'].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "-")
    
    print(display_df.to_string())
    
    # Save to CSV
    output_file = SUMMARY_DIR / "matrix_lgd_no_hpo.csv"
    lgd_matrix_no_hpo.to_csv(output_file)
    print(f"\n✅ Saved to: {output_file.name}")
else:
    print("⚠️  Could not create NO_HPO matrix")

### 📊 B1.2 HPO Performance Matrix

In [ ]:
print("\n" + "=" * 80)
print("  LGD - HPO PERFORMANCE MATRIX (R²)")
print("=" * 80)

lgd_matrix_hpo = create_performance_matrix(lgd_agg, 'R2', hpo_mode='HPO')

if lgd_matrix_hpo is not None:
    # Add summary statistics
    lgd_matrix_hpo['MEAN'] = lgd_matrix_hpo.mean(axis=1)
    lgd_matrix_hpo['STD'] = lgd_agg[lgd_agg['hpo_mode']=='HPO'].groupby('method')['R2_std'].mean()
    
    print(f"\n📊 Matrix shape: {lgd_matrix_hpo.shape[0]} methods × {lgd_matrix_hpo.shape[1]-2} datasets\n")
    
    # Display with nice formatting
    display_df = lgd_matrix_hpo.copy()
    
    # Format numeric columns
    for col in display_df.columns:
        if col not in ['MEAN', 'STD']:
            display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "-")
    display_df['MEAN'] = display_df['MEAN'].apply(lambda x: f"{x:.4f}")
    display_df['STD'] = display_df['STD'].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "-")
    
    print(display_df.to_string())
    
    # Save to CSV
    output_file = SUMMARY_DIR / "matrix_lgd_hpo.csv"
    lgd_matrix_hpo.to_csv(output_file)
    print(f"\n✅ Saved to: {output_file.name}")
else:
    print("⚠️  Could not create HPO matrix")

### 📊 B1.3 HPO Improvement Matrix (HPO - NO_HPO)

In [ ]:
print("\n" + "=" * 80)
print("  LGD - HPO IMPROVEMENT MATRIX (HPO - NO_HPO)")
print("=" * 80)

if lgd_matrix_no_hpo is not None and lgd_matrix_hpo is not None:
    # Remove summary columns for subtraction
    no_hpo_data = lgd_matrix_no_hpo.drop(columns=['MEAN', 'STD'], errors='ignore')
    hpo_data = lgd_matrix_hpo.drop(columns=['MEAN', 'STD'], errors='ignore')
    
    # Calculate difference
    lgd_matrix_diff = hpo_data - no_hpo_data
    
    # Add summary statistics
    lgd_matrix_diff['MEAN'] = lgd_matrix_diff.mean(axis=1)
    lgd_matrix_diff['ABS_MEAN'] = lgd_matrix_diff.drop(columns='MEAN').abs().mean(axis=1)
    
    # Sort by mean improvement
    lgd_matrix_diff = lgd_matrix_diff.sort_values('MEAN', ascending=False)
    
    print(f"\n📊 Matrix shape: {lgd_matrix_diff.shape[0]} methods × {lgd_matrix_diff.shape[1]-2} datasets\n")
    print("💡 Positive values = HPO better, Negative = NO_HPO better\n")
    
    # Display with nice formatting
    display_df = lgd_matrix_diff.copy()
    
    # Format numeric columns with color indicators
    for col in display_df.columns:
        if col not in ['MEAN', 'ABS_MEAN']:
            display_df[col] = display_df[col].apply(
                lambda x: f"{x:+.4f}" if pd.notna(x) else "-"
            )
    display_df['MEAN'] = display_df['MEAN'].apply(lambda x: f"{x:+.4f}")
    display_df['ABS_MEAN'] = display_df['ABS_MEAN'].apply(lambda x: f"{x:.4f}")
    
    print(display_df.to_string())
    
    # Summary statistics
    print("\n" + "-" * 80)
    print("📈 HPO Impact Summary:")
    n_improvements = (lgd_matrix_diff.drop(columns=['MEAN', 'ABS_MEAN']) > 0).sum().sum()
    n_degradations = (lgd_matrix_diff.drop(columns=['MEAN', 'ABS_MEAN']) < 0).sum().sum()
    n_total = lgd_matrix_diff.drop(columns=['MEAN', 'ABS_MEAN']).notna().sum().sum()
    
    print(f"   Improvements: {n_improvements}/{n_total} ({100*n_improvements/n_total:.1f}%)")
    print(f"   Degradations: {n_degradations}/{n_total} ({100*n_degradations/n_total:.1f}%)")
    print(f"   Avg improvement: {lgd_matrix_diff['MEAN'].mean():+.4f}")
    print(f"   Max improvement: {lgd_matrix_diff.drop(columns=['MEAN', 'ABS_MEAN']).max().max():+.4f}")
    print(f"   Max degradation: {lgd_matrix_diff.drop(columns=['MEAN', 'ABS_MEAN']).min().min():+.4f}")
    
    # Save to CSV
    output_file = SUMMARY_DIR / "matrix_lgd_hpo_improvement.csv"
    lgd_matrix_diff.to_csv(output_file)
    print(f"\n✅ Saved to: {output_file.name}")
else:
    print("⚠️  Could not create difference matrix (missing NO_HPO or HPO data)")

## 📊 B2. Performance Bar Plots - LGD

In [ ]:
# =============================================================================
# LGD: BAR PLOT COMPARING NO_HPO (BLUE) vs HPO (RED OUTLINE)
# =============================================================================

print("\n" + "=" * 80)
print("  LGD - AVERAGE R² BAR PLOT (NO_HPO vs HPO)")
print("=" * 80)

if not lgd_agg.empty:
    # Calculate average performance per method for each HPO mode
    no_hpo_means = lgd_agg[lgd_agg['hpo_mode']=='NO_HPO'].groupby('method')['R2_mean'].mean()
    hpo_means = lgd_agg[lgd_agg['hpo_mode']=='HPO'].groupby('method')['R2_mean'].mean()
    
    # Combine into DataFrame
    comparison_df = pd.DataFrame({
        'NO_HPO': no_hpo_means,
        'HPO': hpo_means
    }).fillna(0)
    
    # Sort by HPO performance (descending)
    comparison_df = comparison_df.sort_values('HPO', ascending=False)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(16, 10))
    
    x = np.arange(len(comparison_df))
    width = 0.7
    
    # Plot NO_HPO as filled blue bars
    bars_no_hpo = ax.bar(
        x, comparison_df['NO_HPO'], 
        width, 
        label='NO_HPO',
        color='steelblue',
        alpha=0.8,
        edgecolor='black',
        linewidth=1.5
    )
    
    # Plot HPO as red outline bars
    bars_hpo = ax.bar(
        x, comparison_df['HPO'], 
        width, 
        label='HPO',
        color='none',
        edgecolor='red',
        linewidth=3,
        linestyle='-'
    )
    
    # Customize
    ax.set_xlabel('Method', fontsize=14, fontweight='bold')
    ax.set_ylabel('Average R²', fontsize=14, fontweight='bold')
    ax.set_title('LGD Performance: NO_HPO (Blue Fill) vs HPO (Red Outline)\nRanked by HPO Performance', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(comparison_df.index, rotation=45, ha='right')
    ax.legend(fontsize=12, loc='lower left')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.5)
    
    # Add value labels on bars
    for i, (idx, row) in enumerate(comparison_df.iterrows()):
        # NO_HPO label
        y_pos = row['NO_HPO'] + (0.02 if row['NO_HPO'] > 0 else -0.02)
        va = 'bottom' if row['NO_HPO'] > 0 else 'top'
        ax.text(i, y_pos, f"{row['NO_HPO']:.3f}", 
                ha='center', va=va, fontsize=9, color='steelblue', fontweight='bold')
        # HPO label (if different from NO_HPO)
        if abs(row['HPO'] - row['NO_HPO']) > 0.001:
            y_pos = row['HPO'] + (0.02 if row['HPO'] > 0 else -0.02)
            va = 'bottom' if row['HPO'] > 0 else 'top'
            ax.text(i, y_pos, f"{row['HPO']:.3f}", 
                    ha='center', va=va, fontsize=9, color='red', fontweight='bold')
    
    plt.tight_layout()
    
    # Save figure
    output_file = FIGURES_DIR / "lgd_barplot_no_hpo_vs_hpo.png"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"\n✅ Saved figure: {output_file.name}")
    
    plt.show()
    
    # Print top performers
    print("\n📊 Top 10 Methods (by HPO):")
    print(comparison_df.head(10).to_string())
    
else:
    print("⚠️  No LGD aggregated data available for bar plot")

## 📊 B3. Statistical Analysis - LGD

In [ ]:
# =============================================================================
# FRIEDMAN TEST (Non-parametric ANOVA)
# =============================================================================

print("\n" + "=" * 80)
print("  LGD - FRIEDMAN TEST (Statistical Significance)")
print("=" * 80)

if lgd_matrix_hpo is not None:
    # Remove summary columns
    data = lgd_matrix_hpo.drop(columns=['MEAN', 'STD'], errors='ignore')
    
    # Run Friedman test
    try:
        statistic, p_value = friedmanchisquare(*[data.loc[method].dropna() for method in data.index])
        
        print(f"\n📊 Friedman Test Results:")
        print(f"   Chi-square statistic: {statistic:.4f}")
        print(f"   p-value: {p_value:.6f}")
        print(f"   Significant? {'YES ✓' if p_value < 0.05 else 'NO ✗'} (α=0.05)")
        
        if p_value < 0.05:
            print("\n   ✅ Methods have significantly different performance!")
            print("   ➡️  Proceed with post-hoc tests (Nemenyi, Wilcoxon-Holm)")
        else:
            print("\n   ⚠️  No significant difference detected")
            print("   ➡️  Post-hoc tests may not be meaningful")
    except Exception as e:
        print(f"⚠️  Could not perform Friedman test: {e}")
else:
    print("⚠️  No HPO matrix available for statistical testing")

In [ ]:
# =============================================================================
# AVERAGE RANKS
# =============================================================================

print("\n" + "=" * 80)
print("  LGD - AVERAGE RANKS ACROSS DATASETS")
print("=" * 80)

if lgd_matrix_hpo is not None:
    # Remove summary columns
    data = lgd_matrix_hpo.drop(columns=['MEAN', 'STD'], errors='ignore')
    
    # Calculate ranks for each dataset (higher is better, so use descending)
    ranks = data.rank(axis=0, ascending=False, method='average')
    
    # Calculate average rank per method
    avg_ranks = ranks.mean(axis=1).sort_values()
    
    print(f"\n📊 Average Ranks (1=best, {len(data)}=worst):\n")
    for rank, (method, avg_rank) in enumerate(avg_ranks.items(), 1):
        print(f"   {rank:2d}. {method:20s}  {avg_rank:.2f}")
    
    # Save to CSV
    avg_ranks_df = pd.DataFrame({'method': avg_ranks.index, 'avg_rank': avg_ranks.values})
    output_file = SUMMARY_DIR / "lgd_average_ranks_hpo.csv"
    avg_ranks_df.to_csv(output_file, index=False)
    print(f"\n✅ Saved to: {output_file.name}")
else:
    print("⚠️  No HPO matrix available for ranking")

## 📊 B4. PAMA Analysis - LGD

In [ ]:
# =============================================================================
# PAMA (Probability of Achieving Maximal Accuracy)
# =============================================================================

print("\n" + "=" * 80)
print("  LGD - PAMA ANALYSIS (Probability of Best Performance)")
print("=" * 80)

if lgd_matrix_hpo is not None:
    # Remove summary columns
    data = lgd_matrix_hpo.drop(columns=['MEAN', 'STD'], errors='ignore')
    
    # For each dataset, find the best method(s)
    pama_scores = {}
    
    for dataset in data.columns:
        dataset_scores = data[dataset].dropna()
        if len(dataset_scores) > 0:
            max_score = dataset_scores.max()
            # Allow ties (within 0.0001 tolerance)
            best_methods = dataset_scores[dataset_scores >= max_score - 0.0001].index.tolist()
            
            # Award points (1 if sole winner, 1/n if tie)
            for method in best_methods:
                pama_scores[method] = pama_scores.get(method, 0) + (1.0 / len(best_methods))
    
    # Convert to percentages
    n_datasets = len(data.columns)
    pama_df = pd.DataFrame([
        {'method': method, 'wins': score, 'pama': 100 * score / n_datasets}
        for method, score in pama_scores.items()
    ]).sort_values('pama', ascending=False)
    
    print(f"\n📊 PAMA Scores (across {n_datasets} datasets):\n")
    for idx, row in pama_df.iterrows():
        bar = '█' * int(row['pama'] / 2)  # Visual bar
        print(f"   {row['method']:20s}  {row['wins']:5.1f} wins  {row['pama']:5.1f}%  {bar}")
    
    # Create bar plot
    fig, ax = plt.subplots(figsize=(14, 8))
    
    colors = plt.cm.RdYlGn(pama_df['pama'] / pama_df['pama'].max())
    bars = ax.barh(range(len(pama_df)), pama_df['pama'], color=colors, edgecolor='black', linewidth=1.5)
    
    ax.set_yticks(range(len(pama_df)))
    ax.set_yticklabels(pama_df['method'])
    ax.set_xlabel('PAMA (%)', fontsize=14, fontweight='bold')
    ax.set_title('LGD: Probability of Achieving Maximal Accuracy (PAMA)\nHow often each method wins', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.invert_yaxis()
    
    # Add value labels
    for i, (idx, row) in enumerate(pama_df.iterrows()):
        ax.text(row['pama'] + 1, i, f"{row['pama']:.1f}%", 
                va='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    
    # Save figure
    output_file = FIGURES_DIR / "lgd_pama_analysis.png"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"\n✅ Saved figure: {output_file.name}")
    
    plt.show()
    
    # Save to CSV
    output_file = SUMMARY_DIR / "lgd_pama_scores.csv"
    pama_df.to_csv(output_file, index=False)
    print(f"✅ Saved data: {output_file.name}")
    
else:
    print("⚠️  No HPO matrix available for PAMA analysis")

---
## 🎯 Summary & Conclusions

This notebook provided comprehensive analysis of Experiment1 results including:

✅ **Performance Matrices**: NO_HPO, HPO, and Improvement matrices for both PD and LGD  
✅ **Visual Comparisons**: Bar plots showing HPO vs NO_HPO performance  
✅ **Statistical Testing**: Friedman tests and average rankings  
✅ **PAMA Analysis**: Win rates showing which methods excel most often  

All results and figures have been saved to:
- **Matrices**: `results/experiment1/summary/matrix_*.csv`
- **Figures**: `results/experiment1/figures/*.png`

---

**Next Steps**:
- Add critical difference diagrams (Nemenyi post-hoc test)
- Effect size calculations (Cohen's d)
- Method category comparisons (Classical vs Deep Learning vs Foundation)
- Dataset difficulty analysis